# SQLSage on Google Colab (GPU)

Pipeline: **install → Space liveness → W&B → rollouts → GRPO training → plots**.

1. **Runtime → Change runtime type → GPU** (T4 works for 1.5B 4-bit).
2. **Colab secrets** (🔑): `WANDB_API_KEY` (optional), `SQLSAGE_ENV_URL` (required for real env), `HF_TOKEN` (Hub upload), `WANDB_ENTITY` (optional).
3. **Repo** cloned to `/content/SQLSage`. If your code uses `pyproject.toml` only under `sqlsage-env/`, Cell 1 `cd`s there automatically.
4. The Hugging Face Space must implement **`GET /health`** and **`POST /reset`** (included in the SQLSage app). Cold-start can take many minutes.
5. T4 defaults: `max_seq=1024`, `max_completion=256`, `num_generations=4`, `batch=2`, `grad_accum=8`, `epochs=1` in the GRPO cell (override `SQLSAGE_NUM_TRAIN_EPOCHS=3` for a full run).


In [ ]:
# Cell 1 — GPU check, clone repo, install package

# Optional: os.environ["SQLSAGE_GIT_URL"] = "https://github.com/YOU/SQLSage.git"

%pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
%pip install -q trl openenv-core wandb transformers datasets accelerate huggingface_hub requests "psycopg2-binary"

import os, pathlib, sys, subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime → Change runtime type → T4 (or A100).")

print("Device:", torch.cuda.get_device_name(0), "| CUDA:", torch.version.cuda)

REPO = pathlib.Path("/content/SQLSage")
URL = os.environ.get("SQLSAGE_GIT_URL", "https://github.com/neelshingavi/SQLSage.git")
if not REPO.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
else:
    print("Reusing", REPO)

if (REPO / "pyproject.toml").is_file():
    os.chdir(REPO)
elif (REPO / "sqlsage-env" / "pyproject.toml").is_file():
    os.chdir(REPO / "sqlsage-env")
else:
    raise FileNotFoundError("pyproject.toml not found at /content/SQLSage or .../sqlsage-env")

ROOT = pathlib.Path().resolve()
print("Working directory:", ROOT)
%pip install -q -e '.[training,plots]'

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
pathlib.Path("results").mkdir(exist_ok=True)
print("OK: sqlsage package ready.")


In [ ]:
# Cell 2 — Colab secrets, W&B, Space liveness ( /health, /docs, or /reset )

import os, sys, time, requests, wandb

try:
    from google.colab import userdata
    for key in ("WANDB_API_KEY", "WANDB_ENTITY", "SQLSAGE_ENV_URL", "HF_TOKEN"):
        try:
            v = userdata.get(key)
        except Exception:
            v = None
        if v and not os.environ.get(key, "").strip():
            os.environ[key] = v.strip()
except ImportError:
    pass

if os.environ.get("HF_TOKEN", "").strip():
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"].strip()

os.environ.setdefault("SQLSAGE_ENV_URL", "https://adity00-sqlsage-env.hf.space")
os.environ.setdefault("WANDB_PROJECT", "sqlsage-grpo")

base = os.environ["SQLSAGE_ENV_URL"].rstrip("/")
print("SQLSAGE_ENV_URL =", base)

if os.environ.get("WANDB_API_KEY", "").strip():
    try:
        wandb.login()
        print("W&B: logged in")
    except Exception as e:
        print("W&B login warning:", e)
else:
    os.environ["WANDB_MODE"] = "offline"
    print("W&B: offline (add Secret WANDB_API_KEY to sync to cloud)")

def space_ready(url: str) -> None:
    read_timeout = int(os.environ.get("SQLSAGE_HEALTH_READ_TIMEOUT", "120"))
    max_wait = int(os.environ.get("SQLSAGE_HEALTH_MAX_WAIT", "600"))
    poll = int(os.environ.get("SQLSAGE_HEALTH_POLL_S", "8"))
    deadline = time.time() + max_wait
    n = 0
    while time.time() < deadline:
        n += 1
        for path in ("/health", "/docs", "/openapi.json"):
            try:
                r = requests.get(url + path, timeout=read_timeout)
                if r.status_code == 200:
                    print("Space OK:", path, "->", r.status_code)
                    r2 = requests.post(url + "/reset", json={}, timeout=180)
                    print("/reset", r2.status_code, (r2.text or "")[:200])
                    if r2.status_code == 200:
                        ob = (r2.json() or {}).get("observation", {})
                        print("task_level", ob.get("task_level"), "step", ob.get("step_count"))
                    return
            except requests.RequestException as e:
                print("Try", n, path, e)
        time.sleep(poll)
    raise RuntimeError("Space not reachable. Open the .hf.space URL in a browser, wait, re-run.")

space_ready(base)
print("— Ready for rollouts & training —")


In [ ]:
# Cell 3 — Rollout smoke + baseline JSONL (identity policy) → W&B

import os, sys, subprocess, pathlib

base = os.environ["SQLSAGE_ENV_URL"].rstrip("/")
proj = os.environ.get("WANDB_PROJECT", "sqlsage-grpo")
name = os.environ.get("WANDB_RUN_NAME_ROLLOUT", "colab-baseline")
ep_smoke = int(os.environ.get("SQLSAGE_ROLLOUT_SMOKE_EPISODES", "5"))
ep_base = int(os.environ.get("SQLSAGE_ROLLOUT_BASELINE_EPISODES", "50"))
envp = os.environ.copy()

def run_cmd(args):
    print("$", " ".join(args))
    subprocess.check_call(args, env=envp, cwd=pathlib.Path().resolve())

run_cmd([sys.executable, "scripts/rollout_wandb.py", "--episodes", str(ep_smoke), "--base-url", base, "--project", proj, "--run-name", f"{name}-smoke", "--out-jsonl", "results/smoke.jsonl"])
run_cmd([sys.executable, "scripts/rollout_wandb.py", "--episodes", str(ep_base), "--policy", "identity", "--base-url", base, "--project", proj, "--run-name", f"{name}-baseline", "--out-jsonl", "results/baseline.jsonl"])
if pathlib.Path("results").is_dir():
    for p in sorted(pathlib.Path("results").glob("*.jsonl")):
        print(p, p.stat().st_size, "bytes")


In [ ]:
# Cell 4 — Full GRPO (train.run_training) — T4-tuned via env (see also train.py)

import os
os.environ["SQLSAGE_MAX_SEQ_LENGTH"] = "1024"
os.environ["SQLSAGE_MAX_COMPLETION_LENGTH"] = "256"
os.environ["SQLSAGE_NUM_GENERATIONS"] = "4"
os.environ["SQLSAGE_PER_DEVICE_TRAIN_BATCH_SIZE"] = "2"
os.environ["SQLSAGE_GRADIENT_ACCUMULATION_STEPS"] = "8"
# 1 = smoke, 3 = full run
os.environ["SQLSAGE_NUM_TRAIN_EPOCHS"] = os.environ.get("SQLSAGE_NUM_TRAIN_EPOCHS", "1")
os.environ["SQLSAGE_GRPO_TEMPERATURE"] = "0.9"
os.environ["SQLSAGE_OUTPUT_DIR"] = "sqlsage-grpo"
os.environ["SQLSAGE_MERGED_DIR"] = "sqlsage-grpo-merged"
os.environ["SQLSAGE_MODEL_NAME"] = os.environ.get("SQLSAGE_MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
os.environ["WANDB_RUN_NAME"] = os.environ.get("WANDB_GRPO_RUN_NAME", "colab-grpo-t4")

assert os.environ.get("SQLSAGE_ENV_URL", "").strip(), "Set SQLSAGE_ENV_URL (Colab Secret or above)"

from train import run_training
print(run_training())


In [ ]:
# Cell 5 — Plots, artifact paths, optional download

!python plots/generate_plots.py
!ls -la plots/*.png 2>/dev/null; ls -la sqlsage-grpo/ sqlsage-grpo-merged/ 2>/dev/null; true
# from google.colab import files; files.download("plots/reward_curve.png")
# !python scripts/push_model_to_hub.py --folder ./sqlsage-grpo-merged --repo-id USER/sqlsage-merged
print("LoRA: ./sqlsage-grpo | merged 16bit: ./sqlsage-grpo-merged")
